In [1]:
import numpy as np 

from minitorch.attention import MultiHeadAttention
from minitorch.activations import GELU
from minitorch.dataloader import DataLoader, TensorDataset
from minitorch.embeddings import EmbeddingLayer
from minitorch.tensor import Tensor
from minitorch.tokenizers import CharTokenizer
from minitorch.training import Trainer, CosineScheduler
from minitorch.losses import SoftMaxCrossEntropy
from minitorch.nn import Linear, Module, LayerNormalization, Dropout, Parameter, Sequential
from minitorch.optim import SGD, AdamW
from minitorch.transformer import GPT

In [2]:
text_path = "C:\\Users\\User\\Downloads\\Swahili_Corpus_combined.txt"
with open(text_path, 'r', encoding='utf-8') as f:
    text = f.read()
    
text = text[:100000]
#* split into training and val sets
n = 0.95
text_len = len(text)
chars = sorted(set(text))

#* tokenize the text
tokenizer = CharTokenizer() #* character level tokenization
tokenizer.build_vocab([text])
vocab_size = tokenizer.vocab_size

#* get the train and val loaders
data = Tensor(tokenizer.encode(text))
train_data = data[:int(n*data.shape[0])]
val_data = data[int(n*data.shape[0]):]

In [3]:
max_epochs = 100
eval_iters = 10
context_length, batch_size, embed_dim = 12,8,64

In [4]:
#* prepare data and data loaders
def prepare_data(dataset: Tensor, context_length: int, stride:int =1) -> tuple[Tensor, Tensor]:
    start = range(0, dataset.shape[0] - context_length, stride)
    x = np.stack([dataset[i:i+context_length].data for i in start])
    y = np.stack([dataset[i+1:i+1+context_length].data for i in start])
    return Tensor(x), Tensor(y)


train_features, train_targets = prepare_data(train_data, context_length)
val_features, val_targets = prepare_data(val_data, context_length)


In [ ]:
train_loader = DataLoader(
    TensorDataset([train_features, train_targets]),
    batch_size=8, shuffle=True)
val_loader = DataLoader(
    TensorDataset([val_features, val_targets]),
    batch_size=8, shuffle=False)


In [6]:
#* instantiate the model, optimizer, scheduler, criterion and the trainer class
gpt = GPT(vocab_size=vocab_size, embed_dim=embed_dim, num_heads=8,
            num_layers=6, max_seq_len=context_length, dropout=0.2)

optimizer = AdamW(gpt.parameters(), weight_decay=0.1)
scheduler = CosineScheduler(max_lr=1e-3, min_lr=4e-5, total_epochs=max_epochs)
loss_fn = SoftMaxCrossEntropy()
trainer = Trainer(gpt, optimizer, loss_fn, scheduler=scheduler, grad_clip_norm=1.0)

In [7]:

# Obtain one batch
features, targets = next(iter(train_loader))

# Disable dropout
gpt.eval(mode=False)

# Use a fixed learning rate
optimizer.lr = 1e-3

for step in range(500):
    optimizer.zero_grad()
    logits = gpt(features)
    logits = logits.reshape(-1,vocab_size)
    loss = loss_fn(logits, targets.reshape(-1))
    loss.backward()
    optimizer.step()

    if step % 25 == 0:
        print(f"Step {step:4d} | Loss {float(loss.data):.6f}")



Step    0 | Loss 4.460286
Step   25 | Loss 0.068664
Step   50 | Loss 0.039503
Step   75 | Loss 0.035735
Step  100 | Loss 0.034273
Step  125 | Loss 0.033390
Step  150 | Loss 0.032771
Step  175 | Loss 0.032308
Step  200 | Loss 0.031949
Step  225 | Loss 0.031662
Step  250 | Loss 0.031428
Step  275 | Loss 0.031232
Step  300 | Loss 0.031068
Step  325 | Loss 0.030926
Step  350 | Loss 0.030803
Step  375 | Loss 0.030696
Step  400 | Loss 0.030601
Step  425 | Loss 0.030516
Step  450 | Loss 0.030440
Step  475 | Loss 0.030371


In [ ]:

# #* ###########################################################
#* training loop
print("Epoch          | Training Loss         | Validation Loss | ")
print("-" * 60)

for i in range(max_epochs):        
    training_loss = trainer.train_epoch(train_loader, accumulation_steps=2)
    validation_loss,_ = trainer.evaluate(val_loader)

    if (i+1) % eval_iters == 0 or (i+1) == 1:
        print(f'{i+1}             |   {training_loss:.4f}             | {validation_loss:.4f}')
        

Epoch          | Training Loss         | Validation Loss | 
------------------------------------------------------------
1             |   1.6372             | 1.6215
